## Lab 3
### Part 1: Dealing with overfitting

Today we work with [Fashion-MNIST dataset](https://github.com/zalandoresearch/fashion-mnist) (*hint: it is available in `torchvision`*).

Your goal for today:
1. Train a FC (fully-connected) network that achieves >= 0.885 test accuracy.
2. Cause considerable overfitting by modifying the network (e.g. increasing the number of network parameters and/or layers) and demonstrate in in the appropriate way (e.g. plot loss and accurasy on train and validation set w.r.t. network complexity).
3. Try to deal with overfitting (at least partially) by using regularization techniques (Dropout/Batchnorm/...) and demonstrate the results.

__Please, write a small report describing your ideas, tries and achieved results in the end of this file.__

*Note*: Tasks 2 and 3 are interrelated, in task 3 your goal is to make the network from task 2 less prone to overfitting. Task 1 is independent from 2 and 3.

*Note 2*: We recomment to use Google Colab or other machine with GPU acceleration.

In [33]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchsummary
from IPython.display import clear_output
from matplotlib import pyplot as plt
from matplotlib.pyplot import figure
import numpy as np
import os


device = "cuda:0" if torch.cuda.is_available() else "cpu"

In [34]:
# Technical function
def mkdir(path):
    if not os.path.exists(root_path):
        os.mkdir(root_path)
        print("Directory", path, "is created!")
    else:
        print("Directory", path, "already exists!")


root_path = "fmnist"
mkdir(root_path)

Directory fmnist already exists!


In [35]:
download = True
train_transform = transforms.ToTensor()
test_transform = transforms.ToTensor()
transforms.Compose((transforms.ToTensor()))


fmnist_dataset_train = torchvision.datasets.FashionMNIST(
    root_path, train=True, transform=train_transform, target_transform=None, download=download
)
fmnist_dataset_test = torchvision.datasets.FashionMNIST(
    root_path, train=False, transform=test_transform, target_transform=None, download=download
)

100%|██████████| 26.4M/26.4M [00:13<00:00, 2.02MB/s]


Extracting fmnist/FashionMNIST/raw/train-images-idx3-ubyte.gz to fmnist/FashionMNIST/raw



100%|██████████| 29.5k/29.5k [00:00<00:00, 81.9kB/s]


Extracting fmnist/FashionMNIST/raw/train-labels-idx1-ubyte.gz to fmnist/FashionMNIST/raw



100%|██████████| 4.42M/4.42M [00:02<00:00, 1.87MB/s]


Extracting fmnist/FashionMNIST/raw/t10k-images-idx3-ubyte.gz to fmnist/FashionMNIST/raw



100%|██████████| 5.15k/5.15k [00:00<00:00, 626kB/s]

Extracting fmnist/FashionMNIST/raw/t10k-labels-idx1-ubyte.gz to fmnist/FashionMNIST/raw



In [36]:
train_loader = torch.utils.data.DataLoader(
    fmnist_dataset_train, batch_size=128, shuffle=True, num_workers=2
)
test_loader = torch.utils.data.DataLoader(
    fmnist_dataset_test, batch_size=256, shuffle=False, num_workers=2
)

In [37]:
len(fmnist_dataset_test)

10000

In [38]:
for img, label in train_loader:
    print(img.shape)
    #     print(img)
    print(label.shape)
    print(label.size(0))
    break

torch.Size([128, 1, 28, 28])
torch.Size([128])
128


### Task 1
Train a network that achieves $\geq 0.885$ test accuracy. It's fine to use only Linear (`nn.Linear`) layers and activations/dropout/batchnorm. Convolutional layers might be a great use, but we will meet them a bit later.

In [73]:
class TinyNeuralNetwork(nn.Module):
    def __init__(self, input_shape=28 * 28, num_classes=10, input_channels=1):
        super(self.__class__, self).__init__()
        self.model = nn.Sequential(
            nn.Flatten(),  # This layer converts image into a vector to use Linear layers afterwards
            # Your network structure comes here
            nn.Linear(input_shape, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, inp):
        out = self.model(inp)
        return out

In [74]:
torchsummary.summary(TinyNeuralNetwork().to(device), (28 * 28,))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
           Flatten-1                  [-1, 784]               0
            Linear-2                  [-1, 512]         401,920
              ReLU-3                  [-1, 512]               0
            Linear-4                  [-1, 128]          65,664
              ReLU-5                  [-1, 128]               0
            Linear-6                   [-1, 10]           1,290
Total params: 468,874
Trainable params: 468,874
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.02
Params size (MB): 1.79
Estimated Total Size (MB): 1.81
----------------------------------------------------------------


Your experiments come here:

In [75]:
import torch.optim as optim

In [76]:
model = TinyNeuralNetwork().to(device)
opt = optim.Adam(model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()

# Your experiments, training and validation loops here
for epoch in range(15):
    model.train()
    train_acc, train_loss = 0, 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        preds = model(x)
        loss = loss_func(preds, y)
        loss.backward()
        opt.step()
        train_loss += loss.item()
        train_acc += (preds.argmax(1) == y).sum().item()
    
    train_acc /= len(train_loader.dataset)
    print(f"Epoch {epoch + 1}, Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}")

model.eval()
test_acc = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        preds = model(x)
        test_acc += (preds.argmax(1) == y).sum().item()

test_acc /= len(test_loader.dataset)
print(f"Test Accuracy: {test_acc:.4f}")

Epoch 1, Loss: 253.4276, Accuracy: 0.8097
Epoch 2, Loss: 174.3245, Accuracy: 0.8662
Epoch 3, Loss: 156.8123, Accuracy: 0.8774
Epoch 4, Loss: 143.3383, Accuracy: 0.8877
Epoch 5, Loss: 135.6648, Accuracy: 0.8929
Epoch 6, Loss: 129.0224, Accuracy: 0.8974
Epoch 7, Loss: 121.6173, Accuracy: 0.9027
Epoch 8, Loss: 116.0358, Accuracy: 0.9074
Epoch 9, Loss: 111.3638, Accuracy: 0.9104
Epoch 10, Loss: 106.1608, Accuracy: 0.9139
Epoch 11, Loss: 101.0415, Accuracy: 0.9190
Epoch 12, Loss: 96.8651, Accuracy: 0.9226
Epoch 13, Loss: 94.1569, Accuracy: 0.9233
Epoch 14, Loss: 90.2990, Accuracy: 0.9265
Epoch 15, Loss: 87.0705, Accuracy: 0.9290
Test Accuracy: 0.8898


Эксперимен сразу же дал необхождимое количество точности на тестовых выборках, видимо это благодаря количеству эпох и нескольких слоев в сети. После я попытался изменить количество эпох до 20, после чего точность достигла 0.9, но я решил оставить все как есть

### Task 2: Overfit it.
Build a network that will overfit to this dataset. Demonstrate the overfitting in the appropriate way (e.g. plot loss and accurasy on train and test set w.r.t. network complexity).

*Note:* you also might decrease the size of `train` dataset to enforce the overfitting and speed up the computations.

In [63]:
class OverfittingNeuralNetwork(nn.Module):
    def __init__(self, input_shape=28 * 28, num_classes=10, input_channels=1):
        super(self.__class__, self).__init__()
        self.model = nn.Sequential(
            nn.Flatten(),  # This layer converts image into a vector to use Linear layers afterwards
            # Your network structure comes here
            nn.Linear(input_shape, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, inp):
        out = self.model(inp)
        return out

In [64]:
torchsummary.summary(OverfittingNeuralNetwork().to(device), (28 * 28,))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
           Flatten-1                  [-1, 784]               0
            Linear-2                  [-1, 512]         401,920
              ReLU-3                  [-1, 512]               0
            Linear-4                  [-1, 256]         131,328
              ReLU-5                  [-1, 256]               0
            Linear-6                  [-1, 128]          32,896
              ReLU-7                  [-1, 128]               0
            Linear-8                   [-1, 10]           1,290
Total params: 567,434
Trainable params: 567,434
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.02
Params size (MB): 2.16
Estimated Total Size (MB): 2.19
----------------------------------------------------------------


In [79]:
train_dataset_red = torch.utils.data.Subset(fmnist_dataset_train, range(5000))
test_dataset_red = torch.utils.data.Subset(fmnist_dataset_test, range(5000))

train_loader_red = torch.utils.data.DataLoader(
    train_dataset_red, batch_size=128, shuffle=True, num_workers=2
)
test_loader_red = torch.utils.data.DataLoader(
    test_dataset_red, batch_size=256, shuffle=False, num_workers=2
)

In [80]:
model = OverfittingNeuralNetwork().to(device)
opt = optim.Adam(model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()

# Your experiments, training and validation loops here
for epoch in range(15):
    model.train()
    train_acc, train_loss = 0, 0
    for x, y in train_loader_red:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        preds = model(x)
        loss = loss_func(preds, y)
        loss.backward()
        opt.step()
        train_loss += loss.item()
        train_acc += (preds.argmax(1) == y).sum().item()
    
    train_acc /= len(train_loader_red.dataset)
    print(f"Epoch {epoch + 1}, Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}")

model.eval()
test_acc = 0
with torch.no_grad():
    for x, y in test_loader_red:
        x, y = x.to(device), y.to(device)
        preds = model(x)
        test_acc += (preds.argmax(1) == y).sum().item()

test_acc /= len(test_loader_red.dataset)
print(f"Test Accuracy: {test_acc:.4f}")

Epoch 1, Loss: 48.8127, Accuracy: 0.5534
Epoch 2, Loss: 27.1504, Accuracy: 0.7504
Epoch 3, Loss: 22.5225, Accuracy: 0.7992
Epoch 4, Loss: 20.0906, Accuracy: 0.8164
Epoch 5, Loss: 18.6170, Accuracy: 0.8296
Epoch 6, Loss: 17.0857, Accuracy: 0.8478
Epoch 7, Loss: 16.5516, Accuracy: 0.8516
Epoch 8, Loss: 15.1279, Accuracy: 0.8646
Epoch 9, Loss: 14.1644, Accuracy: 0.8780
Epoch 10, Loss: 14.2295, Accuracy: 0.8670
Epoch 11, Loss: 11.9984, Accuracy: 0.8926
Epoch 12, Loss: 11.7415, Accuracy: 0.8940
Epoch 13, Loss: 11.9354, Accuracy: 0.8890
Epoch 14, Loss: 10.8347, Accuracy: 0.9038
Epoch 15, Loss: 12.4600, Accuracy: 0.8890
Test Accuracy: 0.8278


Видно что из за добавления дополнительных слоев и уменьшение выборки повлияли на точность модели. она упала до 0.82

### Task 3: Fix it.
Fix the overfitted network from the previous step (at least partially) by using regularization techniques (Dropout/Batchnorm/...) and demonstrate the results. 

In [84]:
class FixedNeuralNetwork(nn.Module):
    def __init__(self, input_shape=28 * 28, num_classes=10, input_channels=1):
        super(self.__class__, self).__init__()
        self.model = nn.Sequential(
            nn.Flatten(),  # This layer converts image into a vector to use Linear layers afterwards
            # Your network structure comes here
            nn.Linear(input_shape, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.2),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )

    def forward(self, inp):
        out = self.model(inp)
        return out

In [85]:
torchsummary.summary(FixedNeuralNetwork().to(device), (28 * 28,))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
           Flatten-1                  [-1, 784]               0
            Linear-2                  [-1, 512]         401,920
              ReLU-3                  [-1, 512]               0
       BatchNorm1d-4                  [-1, 512]           1,024
           Dropout-5                  [-1, 512]               0
            Linear-6                  [-1, 256]         131,328
              ReLU-7                  [-1, 256]               0
       BatchNorm1d-8                  [-1, 256]             512
           Dropout-9                  [-1, 256]               0
           Linear-10                  [-1, 128]          32,896
             ReLU-11                  [-1, 128]               0
      BatchNorm1d-12                  [-1, 128]             256
          Dropout-13                  [-1, 128]               0
           Linear-14                   

In [86]:
model = FixedNeuralNetwork().to(device)
opt = optim.Adam(model.parameters(), lr=0.001)
loss_func = nn.CrossEntropyLoss()

# Your experiments, training and validation loops here
for epoch in range(15):
    model.train()
    train_acc, train_loss = 0, 0
    for x, y in train_loader_red:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        preds = model(x)
        loss = loss_func(preds, y)
        loss.backward()
        opt.step()
        train_loss += loss.item()
        train_acc += (preds.argmax(1) == y).sum().item()
    
    train_acc /= len(train_loader_red.dataset)
    print(f"Epoch {epoch + 1}, Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}")

model.eval()
test_acc = 0
with torch.no_grad():
    for x, y in test_loader_red:
        x, y = x.to(device), y.to(device)
        preds = model(x)
        test_acc += (preds.argmax(1) == y).sum().item()

test_acc /= len(test_loader_red.dataset)
print(f"Test Accuracy: {test_acc:.4f}")

Epoch 1, Loss: 31.5390, Accuracy: 0.7400
Epoch 2, Loss: 20.1792, Accuracy: 0.8272
Epoch 3, Loss: 17.3122, Accuracy: 0.8520
Epoch 4, Loss: 15.7283, Accuracy: 0.8594
Epoch 5, Loss: 15.0997, Accuracy: 0.8686
Epoch 6, Loss: 15.4004, Accuracy: 0.8640
Epoch 7, Loss: 13.9120, Accuracy: 0.8804
Epoch 8, Loss: 13.7846, Accuracy: 0.8802
Epoch 9, Loss: 14.0446, Accuracy: 0.8704
Epoch 10, Loss: 13.7061, Accuracy: 0.8816
Epoch 11, Loss: 13.7101, Accuracy: 0.8760
Epoch 12, Loss: 12.0725, Accuracy: 0.8896
Epoch 13, Loss: 11.8960, Accuracy: 0.8926
Epoch 14, Loss: 11.9261, Accuracy: 0.8966
Epoch 15, Loss: 11.8807, Accuracy: 0.8998
Test Accuracy: 0.8316


### Conclusions:

Благодаря Dropout/Batchnorm мы смогли повысить точность модели до 0.83, эот к сожелению не большой прорыв, но если увеличить количество эпох, этот показатель значительно улучшится